# Goodreads 10M Vibe Recommender — Full 1M+ Books GPU Index Builder
Make sure your Colab runtime is set to **GPU** (`T4 GPU`, `L4 GPU`, or `A100 GPU` with **High-RAM** enabled), then run the cell below.

In [ ]:
import os
import re
import ast
import pickle
import zipfile
import requests
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from sklearn.feature_extraction.text import TfidfVectorizer
from huggingface_hub import hf_hub_download

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

KAGGLE_ZIP_PATH = os.path.join(DATA_DIR, "goodreads_10m.zip")
KAGGLE_URL = (
    "https://www.kaggle.com/api/v1/datasets/download/"
    "bahramjannesarr/goodreads-book-datasets-10m"
)

CSV_PATH = os.path.join(DATA_DIR, "goodreads_books.csv")
EMBEDDINGS_PATH = os.path.join(DATA_DIR, "book_embeddings.npy")
TFIDF_PATH = os.path.join(DATA_DIR, "tfidf_vectorizer.pkl")
TFIDF_MATRIX_PATH = os.path.join(DATA_DIR, "tfidf_matrix.pkl")
OUTPUT_ZIP_PATH = "goodreads_index_files.zip"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

GENRE_KEYWORDS = {
    "Gothic": [
        "gothic", "macabre", "victorian", "brooding",
        "manor", "haunted house", "dracula", "frankenstein"
    ],
    "Horror": [
        "horror", "terrifying", "nightmare", "supernatural", "ghost",
        "haunting", "monster", "undead", "vampire", "zombie", "creepy", "chilling"
    ],
    "Mystery": [
        "mystery", "detective", "murder", "investigation",
        "whodunit", "crime", "clues", "suspect", "sleuth", "disappearance"
    ],
    "Thriller": [
        "thriller", "suspense", "conspiracy", "assassin", "espionage",
        "kidnapped", "stalker", "deadly", "hostage", "psychopath"
    ],
    "Science Fiction": [
        "science fiction", "sci-fi", "spaceship", "galaxy", "alien", "planet",
        "interstellar", "cyborg", "android", "time travel", "martian", "spacecraft"
    ],
    "Cyberpunk": [
        "cyberpunk", "cyberspace", "hacker", "neon", "virtual reality",
        "megacorp", "dystopian future", "artificial intelligence"
    ],
    "Dystopia": [
        "dystopia", "dystopian", "totalitarian", "post-apocalyptic",
        "apocalypse", "wasteland", "regime", "rebellion"
    ],
    "Fantasy": [
        "fantasy", "magic", "magical", "wizard", "dragon", "sorcerer",
        "kingdom", "sword", "prophecy", "fae", "faerie", "elf", "elves", "spell"
    ],
    "Paranormal": [
        "paranormal", "werewolf", "shifter", "vampires", "witches",
        "warlock", "demon", "immortal", "fated mates"
    ],
    "Romance": [
        "romance", "romantic", "love story", "passionate", "heartbreak",
        "lover", "marriage", "wedding", "seduction", "billionaire"
    ],
    "Historical Fiction": [
        "historical", "century", "world war", "victorian", "regency",
        "medieval", "tudor", "ancient", "civil war", "empire"
    ],
    "Philosophy": [
        "philosophy", "philosophical", "existential", "metaphysical",
        "ethics", "consciousness", "stoic", "enlightenment"
    ],
    "Psychology": [
        "psychology", "psychological", "psychiatrist", "asylum",
        "trauma", "madness", "mind", "obsessive", "delusion"
    ],
    "Classics": [
        "classic", "masterpiece", "19th century", "20th century",
        "literary classic", "pulitzer", "nobel"
    ],
    "Nonfiction": [
        "biography", "memoir", "autobiography", "history of",
        "essays", "journalism", "true story", "scientific"
    ],
}

def normalize_key(title: str) -> str:
    t = str(title).lower()
    t = re.sub(r"\s*\(.*?\)\s*", " ", t)
    t = re.sub(r"[^a-z0-9]+", " ", t).strip()
    return t

def parse_genres_list(val):
    if pd.isna(val):
        return []
    try:
        parsed = ast.literal_eval(str(val))
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except Exception:
        pass
    return [x.strip() for x in str(val).split(",") if x.strip()]

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)

# 1. Download Kaggle 10M Dataset
if not (os.path.exists(KAGGLE_ZIP_PATH) and os.path.getsize(KAGGLE_ZIP_PATH) > 400_000_000):
    print("1. Downloading Goodreads 10M dataset from Kaggle...")
    with requests.get(KAGGLE_URL, stream=True) as r:
        r.raise_for_status()
        with open(KAGGLE_ZIP_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024 * 8):
                if chunk:
                    f.write(chunk)
    print("Download complete!")
else:
    print(f"Found existing archive: {KAGGLE_ZIP_PATH}")

# 2. Load Goodreads shelf tags lookup
print("2. Loading Goodreads shelf tags lookup...")
genre_lookup = {}
try:
    hf_csv = hf_hub_download(
        "Eitanli/goodreads", "goodreads_data.csv", repo_type="dataset"
    )
    hf_df = pd.read_csv(hf_csv)
    for _, row in hf_df.iterrows():
        k = normalize_key(row.get("Book", ""))
        glist = parse_genres_list(row.get("Genres", ""))
        if k and glist:
            genre_lookup[k] = ", ".join(glist)
except Exception as e:
    print(f"Warning: Could not load auxiliary genre tags: {e}")

# 3. Read ALL CSV shards (No rating cutoff -> 1.1M+ books!)
print("3. Reading all CSV shards from Goodreads 10M (no rating cutoff)...")
chunks = []
target_cols = [
    "Id", "Name", "Authors", "Rating", "PublishYear",
    "Publisher", "RatingDistTotal", "CountsOfReview", "Language", "Description"
]
valid_langs = ["eng", "en-US", "en-GB", "en-CA", "en"]

with zipfile.ZipFile(KAGGLE_ZIP_PATH, "r") as z:
    for info in z.infolist():
        if info.filename.startswith("book") and info.filename.endswith(".csv"):
            with z.open(info.filename) as f:
                head_df = pd.read_csv(f, nrows=1)
                if "Description" not in head_df.columns:
                    continue
            with z.open(info.filename) as f:
                cols = [c for c in target_cols if c in head_df.columns]
                df_shard = pd.read_csv(f, usecols=cols, low_memory=False)
                lang_mask = df_shard["Language"].fillna("eng").isin(valid_langs)
                df_shard = df_shard[lang_mask & df_shard["Description"].notna()].copy()
                raw_tot = (
                    df_shard["RatingDistTotal"]
                    .astype(str)
                    .str.replace("total:", "", regex=False)
                )
                df_shard["Num_Ratings_Int"] = (
                    pd.to_numeric(raw_tot, errors="coerce").fillna(0).astype(np.int32)
                )
                chunks.append(df_shard)
                print(f"  Loaded {info.filename}: {len(df_shard):,} books")

df = pd.concat(chunks, ignore_index=True)
del chunks
print(f"Combined raw shards: {len(df):,} books with descriptions!")

# 4. Vectorized HTML cleaning & normalization (runs in seconds on 1M+ rows)
print("4. Vectorized cleaning of 1M+ descriptions...")
df["Description"] = (
    df["Description"]
    .astype(str)
    .str.replace(r"<[^>]+>", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
df = df[df["Description"].str.len() >= 30].reset_index(drop=True)

df["Book"] = df["Name"].fillna("Unknown Title").astype(str).str.strip()
df["Author"] = df["Authors"].fillna("Unknown Author").astype(str).str.strip()
df["Avg_Rating"] = pd.to_numeric(df["Rating"], errors="coerce").fillna(0.0).astype(np.float32)
df["URL"] = "https://www.goodreads.com/book/show/" + df["Id"].astype(str)
df = df.drop(columns=["Name", "Authors", "Rating", "RatingDistTotal", "Language"], errors="ignore")

# 5. Vectorized Genre & Vibe Tagging across 1M+ books
print("5. Vectorized genre & vibe tagging across 1M+ books...")
search_blob = (df["Book"] + " " + df["Description"]).str.lower()
matched_cols = []
for genre, words in GENRE_KEYWORDS.items():
    pat = r"\b(?:" + "|".join(words) + r")\b"
    hit = search_blob.str.contains(pat, regex=True, na=False)
    matched_cols.append(np.where(hit, genre, ""))

del search_blob
genres_clean = pd.Series([", ".join(filter(None, row)) for row in zip(*matched_cols)])
genres_clean = genres_clean.replace("", "Fiction")

if genre_lookup:
    norm_keys = (
        df["Book"]
        .str.lower()
        .str.replace(r"\s*\(.*?\)\s*", " ", regex=True)
        .str.replace(r"[^a-z0-9]+", " ", regex=True)
        .str.strip()
    )
    mapped = norm_keys.map(genre_lookup)
    genres_clean = mapped.fillna(genres_clean)

df["Genres_Clean"] = genres_clean.values
df["Genres"] = df["Genres_Clean"]

print(f"Final full Goodreads catalog size: {len(df):,} books!")
df.to_csv(CSV_PATH, index=False)
print(f"Saved {CSV_PATH}")

vibe_docs = (
    "Book Title: " + df["Book"] + " by " + df["Author"] + ". "
    + "Atmosphere, Vibe & Genres: " + df["Genres_Clean"] + ". "
    + "Synopsis: " + df["Description"]
)

# 6. Build TF-IDF Vectorizer & Matrix
print("6. Building TF-IDF lexical vibe matrix for 1M+ books...")
tfidf = TfidfVectorizer(
    max_features=35000,
    stop_words="english",
    ngram_range=(1, 1),
    sublinear_tf=True,
    dtype=np.float32,
)
tfidf_matrix = tfidf.fit_transform(vibe_docs)
with open(TFIDF_PATH, "wb") as f:
    pickle.dump(tfidf, f)
with open(TFIDF_MATRIX_PATH, "wb") as f:
    pickle.dump(tfidf_matrix, f)
print(f"Saved TF-IDF matrix of shape {tfidf_matrix.shape}.")

# 7. Compute Dense Neural Embeddings on GPU (Pre-allocated FP16)
print(f"7. Computing neural semantic embeddings for {len(df):,} books using {MODEL_NAME}...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
if device == "cuda":
    model = model.half()
model.eval()

texts = vibe_docs.tolist()
del vibe_docs
total_books = len(texts)
batch_size = 1024 if device == "cuda" else 128
embeddings_matrix = np.empty((total_books, 384), dtype=np.float16)

with torch.no_grad():
    for i in range(0, total_books, batch_size):
        batch_texts = texts[i : i + batch_size]
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        ).to(device)
        model_output = model(**encoded)
        sentence_embeddings = mean_pooling(model_output, encoded["attention_mask"])
        sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)
        embeddings_matrix[i : i + len(batch_texts)] = (
            sentence_embeddings.cpu().numpy().astype(np.float16)
        )
        if (i // batch_size) % 25 == 0:
            done = min(i + batch_size, total_books)
            print(f"  Encoded {done:,}/{total_books:,} books...")

np.save(EMBEDDINGS_PATH, embeddings_matrix)
print(f"Saved embeddings of shape {embeddings_matrix.shape} to {EMBEDDINGS_PATH}")

# 8. Package into a single ZIP and download
print(f"8. Packaging index files into {OUTPUT_ZIP_PATH}...")
with zipfile.ZipFile(OUTPUT_ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(CSV_PATH, arcname="goodreads_books.csv")
    zf.write(EMBEDDINGS_PATH, arcname="book_embeddings.npy")
    zf.write(TFIDF_PATH, arcname="tfidf_vectorizer.pkl")
    zf.write(TFIDF_MATRIX_PATH, arcname="tfidf_matrix.pkl")

zip_size_mb = os.path.getsize(OUTPUT_ZIP_PATH) / (1024 * 1024)
print(f"Created {OUTPUT_ZIP_PATH} ({zip_size_mb:.1f} MB). Triggering download...")

from google.colab import files
files.download(OUTPUT_ZIP_PATH)